# AOS Open-Loop Reconstruction

Reconstruct the closed-loop PID behavior of the Rubin AOS system,
as validation for subsequent open-loop disturbance reconstruction.

**Author:** Aaron Roodman  
**Date Created:** 2026-03-11

## Table of Contents

1. [Setup](#Setup)
2. [Analysis Code](#Analysis-Code)
3. [Load Data](#Load-Data)
4. [PID Simulation](#PID-Simulation)
5. [Validation Plots](#Validation-Plots)
6. [Multi-Night](#Multi-Night)


<a id="Setup"></a>
## Setup

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.stats import median_abs_deviation

from astropy.stats import biweight_midcorrelation

from lsst.ts.ofc import OFCData, SensitivityMatrix
from lsst.ts.ofc.state_estimator import StateEstimator

import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [ ]:
# --- Location ---
location = 'summit'   # 'summit' or 'usdf'

# --- Parameters ---
day_obs = 20260117
parquet_file = f"nightly_aos_table_{day_obs}.parquet"

# SVD / DOF configuration
ofc_config_dirs = {
    'summit': '/home/aroodman/packages/ts_config_mttcs/MTAOS/v8/ofc',
    'usdf':   '/home/r/roodman/u/LSST/packages/ts_config_mttcs/MTAOS/v8/ofc',
}
ofc_config_dir = ofc_config_dirs[location]
dof_set_name = "standard_22"   # which DOF subset
n_vmodes = 12                  # number of vmodes to simulate

# PID gains — per-vmode arrays (length n_vmodes)
kappaP = np.ones(n_vmodes) * 0.18
kappaP[11] = 0.045   # vmode 12 (1-indexed) has lower P gain

kappaI = np.ones(n_vmodes) * 0.022

kappaD = np.zeros(n_vmodes)

# Integral clamp per vmode
max_integral = np.array([0.001, 0.001, 0.01, 0.01, 3, 3, 3, 4, 1.5, 0.05, 0.1, 10])

# Z4 control setpoint in microns
Z4_setpoint = -0.2

# Zernike source: 'deviation' (OPD minus intrinsic) or 'opd' (full OPD)
zk_column_type = 'deviation'

# Sensitivity matrix rotation option
# True = recompute SVD per image at actual rotation_angle (matches online OFC)
# False = fixed SVD at rotation_angle=0 (faster, current default)
use_per_image_rotation = False

<a id="Analysis-Code"></a>
## Analysis Code


In [ ]:
# --- DOF labels (all 50) ---
labels_50dof = [
    'M2_dz', 'M2_dx', 'M2_dy', 'M2_rx', 'M2_ry',
    'Cam_dz', 'Cam_dx', 'Cam_dy', 'Cam_rx', 'Cam_ry',
    'B1_1', 'B1_2', 'B1_3', 'B1_4', 'B1_5',
    'B1_6', 'B1_7', 'B1_8', 'B1_9', 'B1_10',
    'B1_11', 'B1_12', 'B1_13', 'B1_14', 'B1_15',
    'B1_16', 'B1_17', 'B1_18', 'B1_19', 'B1_20',
    'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5',
    'B2_6', 'B2_7', 'B2_8', 'B2_9', 'B2_10',
    'B2_11', 'B2_12', 'B2_13', 'B2_14', 'B2_15',
    'B2_16', 'B2_17', 'B2_18', 'B2_19', 'B2_20',
]
_label_to_index = {name: i for i, name in enumerate(labels_50dof)}
def label_index(name):
    return _label_to_index[name]

# --- DOF subsets ---
dof_sets = {
    'hexapod_10': list(range(0, 10)),
    'standard_22': list(range(0, 17)) + list(range(30, 35)),
    'extended_30': list(range(0, 20)) + list(range(30, 40)),
    'all_50': list(range(0, 50)),
}

# --- Zernike selection (21 terms: z4-z19, z22-z26, skip z20,z21,z27,z28) ---
zn = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26])
zn_idx = zn - 4   # indices into dense zernike arrays (z4=index 0)
n_zernike = len(zn_idx)

# --- WFS corners ---
sensor_name_list = ['R00_SW0', 'R04_SW0', 'R40_SW0', 'R44_SW0']
n_wfs = len(sensor_name_list)

# --- WFS corner short names (matching Butler column suffixes) ---
# R00=det191, R04=det195, R40=det199, R44=det203
wf_corners = ['R00', 'R04', 'R40', 'R44']

print(f'Zernike terms: {zn} ({n_zernike} terms)')
print(f'Measurement vector size: {n_zernike} x {n_wfs} = {n_zernike * n_wfs}')


In [ ]:
def build_sensitivity_matrix(dof_set_name, dof_sets, zn_idx, sensor_name_list, ofc_config_dir, rotation_angle=0.0):
    """Build sensitivity matrix and compute SVD for a given DOF subset.

    Parameters
    ----------
    dof_set_name : str
        Key into dof_sets dict.
    dof_sets : dict
        Maps name -> list of DOF indices.
    zn_idx : array
        Zernike indices (0-based, relative to z4).
    sensor_name_list : list of str
        WFS sensor names.
    ofc_config_dir : str
        Path to OFC configuration directory.

    Returns
    -------
    svd_result : dict
        Keys: U, s, V, dof_indices, labels, Atilde_sub, norm_vector, A_sub, n_modes
    """
    ofc_data = OFCData('lsst', config_dir=ofc_config_dir)
    ofc_data.configure_controller()
    field_angles = [ofc_data.sample_points[s] for s in sensor_name_list]

    dz_sensitivity_matrix = SensitivityMatrix(ofc_data)
    sens_3d = dz_sensitivity_matrix.evaluate(field_angles, rotation_angle)

    # Select Zernike subset, reshape to 2D
    sens_3d = sens_3d[:, zn_idx, :]
    A_full = sens_3d.reshape((-1, sens_3d.shape[2]))

    # Restrict to chosen DOF subset, then normalize
    dof_indices = dof_sets[dof_set_name]
    A_sub = A_full[:, dof_indices]
    norm_vector = ofc_data.normalization_weights[dof_indices]
    Atilde_sub = A_sub @ np.diag(norm_vector)

    # SVD
    U, s, Vh = np.linalg.svd(Atilde_sub, full_matrices=False)
    V = Vh.T
    labels_sub = [labels_50dof[i] for i in dof_indices]

    print(f'Built sensitivity matrix for DOF set "{dof_set_name}":')
    print(f'  A shape: {A_sub.shape}  (n_meas x n_dof)')
    print(f'  {len(s)} singular values: max={s[0]:.3f}, min={s[-1]:.2e}')

    return dict(
        U=U, s=s, V=V,
        dof_indices=dof_indices,
        labels=labels_sub,
        Atilde_sub=Atilde_sub,
        A_sub=A_sub,
        norm_vector=norm_vector,
        n_modes=len(s),
        ofc_data=ofc_data,
        dz_sensitivity_matrix=dz_sensitivity_matrix,
    )


def build_svd_at_rotation(svd_result, rotation_angle, zn_idx):
    """Recompute SVD at a specific rotation angle using cached objects.

    Parameters
    ----------
    svd_result : dict
        From build_sensitivity_matrix (must include ofc_data, dz_sensitivity_matrix).
    rotation_angle : float
        Rotator angle in degrees.
    zn_idx : array
        Zernike index selection.

    Returns
    -------
    dict with U, s, V, norm_vector, dof_indices
    """
    ofc_data = svd_result['ofc_data']
    dz_sm = svd_result['dz_sensitivity_matrix']
    dof_indices = svd_result['dof_indices']

    field_angles = [ofc_data.sample_points[s] for s in sensor_name_list]
    sens_3d = dz_sm.evaluate(field_angles, rotation_angle)

    # Select Zernike subset, reshape to 2D (all 50 DOFs)
    sens_3d = sens_3d[:, zn_idx, :]
    A_full = sens_3d.reshape((-1, sens_3d.shape[2]))

    # Restrict to DOF subset, then normalize
    A_sub = A_full[:, dof_indices]
    norm_vector = ofc_data.normalization_weights[dof_indices]
    Atilde_sub = A_sub @ np.diag(norm_vector)

    # SVD
    U, s_vals, Vh = np.linalg.svd(Atilde_sub, full_matrices=False)
    V = Vh.T

    return dict(U=U, s=s_vals, V=V, norm_vector=norm_vector, dof_indices=dof_indices)

In [ ]:
def zernikes_to_vmodes(zernikes_per_corner, svd_result, zn_idx,
                       n_vmodes=12, z4_setpoint=0.0, U_override=None):
    """Convert per-corner Zernike deviations to vmode amplitudes.

    Applies Z4 setpoint correction before projecting onto vmodes.
    Input Zernikes should already be deviations (OPD - intrinsic).

    Parameters
    ----------
    zernikes_per_corner : dict
        {corner: array} for each WFS corner (e.g. 'R00', 'R04', ...).
        Dense arrays with index 0 = Z4.
    svd_result : dict
        From build_sensitivity_matrix.
    zn_idx : array
        Indices of the 21 Zernike terms to use (0-based from Z4).
    n_vmodes : int
        Number of vmodes to return.
    z4_setpoint : float
        Z4 control setpoint in microns. Subtracted from Z4 before projection.
    U_override : array, optional
        If provided, use this U matrix instead of svd_result['U'].
        Used for per-image rotation-dependent SVD.

    Returns
    -------
    a : array (n_vmodes,)
        Vmode amplitudes (a_k = U^T_k @ z_corrected).
    z_corrected_per_corner : dict
        {corner: array} corrected Zernikes for each corner.
    """
    z_parts = []
    z_corrected_per_corner = {}
    for corner in wf_corners:
        zk = np.array(zernikes_per_corner[corner], dtype=float).copy()

        # Subtract Z4 setpoint (Z4 is index 0 in the dense array)
        zk[0] -= z4_setpoint

        z_corrected_per_corner[corner] = zk
        z_parts.append(zk[zn_idx])

    z_vec = np.concatenate(z_parts)

    # Project onto vmodes
    U = U_override if U_override is not None else svd_result["U"]
    a = U[:, :n_vmodes].T @ z_vec
    return a, z_corrected_per_corner


def table_to_vmode_timeseries(table, svd_result, zn_idx, n_vmodes=12,
                              z4_setpoint=0.0, zk_column_type='deviation',
                              use_per_image_rotation=False):
    """Convert all exposures in table to vmode timeseries.

    Reads zk_{zk_column_type}_{corner} columns from the table, applies
    Z4 setpoint correction, then projects onto vmodes.

    Parameters
    ----------
    table : DataFrame
        Must have columns zk_{zk_column_type}_R00..R44.
    svd_result : dict
    zn_idx : array
    n_vmodes : int
    z4_setpoint : float
        Z4 control setpoint in microns.
    zk_column_type : str
        'deviation' (OPD minus intrinsic, default) or 'opd' (full OPD).
    use_per_image_rotation : bool
        If True, recompute SVD at each image's rotation_angle.

    Returns
    -------
    vmode_array : array (n_exp, n_vmodes)
        Vmode amplitudes from the SVD decomposition of corrected Zernikes.
    z_corrected_all : dict
        {corner: list of arrays} per-exposure corrected Zernikes.
    s_per_image : array (n_exp, n_vmodes)
        Singular values used for each exposure (varies when use_per_image_rotation=True).
    """
    n_exp = len(table)
    vmode_array = np.zeros((n_exp, n_vmodes))
    s_per_image = np.zeros((n_exp, n_vmodes))
    z_corrected_all = {corner: [] for corner in wf_corners}

    # Safe fallback length for missing data
    n_zk_min = max(zn_idx) + 1

    # Default singular values (fixed rotation)
    s_default = svd_result['s'][:n_vmodes]
    U_default = svd_result['U']

    for i in range(n_exp):
        zk_dict = {}
        for corner in wf_corners:
            col = f"zk_{zk_column_type}_{corner}"
            zk = table.iloc[i][col]
            if zk is None or (isinstance(zk, float) and np.isnan(zk)):
                zk = np.zeros(n_zk_min)
            zk_dict[corner] = np.array(zk, dtype=float)

        # Optionally recompute SVD at this image's rotation angle
        if use_per_image_rotation and 'rotation_angle' in table.columns:
            rot_angle = float(table.iloc[i]['rotation_angle'])
            svd_i = build_svd_at_rotation(svd_result, rot_angle, zn_idx)
            U_i = svd_i['U']
            s_i = svd_i['s'][:n_vmodes]
        else:
            U_i = U_default
            s_i = s_default

        vmode_array[i], z_corr = zernikes_to_vmodes(
            zk_dict, svd_result, zn_idx, n_vmodes,
            z4_setpoint=z4_setpoint, U_override=U_i)
        s_per_image[i] = s_i

        for corner in wf_corners:
            z_corrected_all[corner].append(z_corr[corner])

    if use_per_image_rotation:
        print(f"  Per-image rotation SVD: {n_exp} evaluations")

    return vmode_array, z_corrected_all, s_per_image

In [ ]:
def align_svd_signs(svd_result, dof_state, vmodes_from_table, n_vmodes=12):
    """Align SVD eigenvector signs with the StateEstimator convention.

    The SVD has an inherent sign ambiguity: U[:,k] and V[:,k] can both
    be negated without changing the decomposition. This function flips
    signs to match the table vmodes (computed by StateEstimator).

    Parameters
    ----------
    svd_result : dict
        From build_sensitivity_matrix. Modified in place.
    dof_state : array (n_exp, 50)
        DOF state from table.
    vmodes_from_table : array (n_exp, n_vmodes_table)
        Vmodes from StateEstimator (table['vmodes']).
    n_vmodes : int
        Number of vmodes to align.

    Returns
    -------
    svd_result : dict
        Same dict, with U and V signs aligned.
    """
    V = svd_result["V"]
    norm_sub = svd_result["norm_vector"]  # already subset-restricted to active DOFs
    dof_idx = svd_result["dof_indices"]
    U = svd_result["U"]

    n_exp = len(dof_state)
    n_align = min(n_vmodes, vmodes_from_table.shape[1], V.shape[1])

    # Compute DOF-based vmodes using our SVD
    vmodes_our = np.zeros((n_exp, n_align))
    for i in range(n_exp):
        dof_used = dof_state[i, dof_idx]
        dof_tilde = dof_used / norm_sub
        vmodes_our[i] = (V.T @ dof_tilde)[:n_align]

    # Check correlation and flip where needed
    n_flipped = 0
    for k in range(n_align):
        corr = np.corrcoef(vmodes_from_table[:, k], vmodes_our[:, k])[0, 1]
        if corr < 0:
            V[:, k] *= -1
            U[:, k] *= -1
            n_flipped += 1
            print(f"  Vmode {k+1}: flipped sign (corr was {corr:.4f})")
        else:
            print(f"  Vmode {k+1}: sign OK (corr = {corr:.4f})")

    print(f"Sign alignment complete: flipped {n_flipped}/{n_align} vmodes")
    return svd_result

In [ ]:
# Intrinsic Zernike computation removed — Butler zk_deviation columns
# already have intrinsics subtracted. For diagnostics, intrinsics are
# available in the table's zk_intrinsic_{corner} columns.

In [ ]:
def _pid_step(pv_value, kappaP, kappaI, kappaD, integral, prev_pv, max_integral):
    """Single PID controller step.

    Returns: mv, mv_p, mv_i, mv_d, integral_updated
    """
    mv_p = kappaP * pv_value
    integral += pv_value
    integral = np.clip(integral, -max_integral, max_integral)
    mv_i = kappaI * integral
    mv_d = kappaD * (pv_value - prev_pv)
    mv = mv_p + mv_i + mv_d
    return mv, mv_p, mv_i, mv_d, integral


def simulate_pid_all_vmodes(table, svd_result, zn_idx,
                            kappaP=0.18, kappaI=0.0, kappaD=0.0,
                            max_integral=None, n_vmodes=12,
                            z4_setpoint=0.0, zk_column_type='deviation',
                            use_per_image_rotation=False):
    """Simulate PID control loops for all vmodes.

    For each exposure, compute vmode amplitudes from the 4-corner Zernike
    columns (zk_{zk_column_type}_{corner}) via the SVD, scale by 1/s_k
    to match Rubin's DOF-space vmode convention (v_k = U_k^T z / s_k),
    then run independent PID loops per vmode.

    Parameters
    ----------
    table : DataFrame
        From nightly_tablemaker. Must have zk_{zk_column_type}_R00..R44, seq,
        seq_num_corr, vmodes.
    svd_result : dict
        From build_sensitivity_matrix (with signs aligned).
    zn_idx : array
        Zernike index selection.
    kappaP, kappaI, kappaD : float or array (n_vmodes,)
        PID gains. Scalars are broadcast to all vmodes.
    max_integral : array (n_vmodes,)
        Integral clamp per vmode. If None, defaults to 5.0 for all vmodes.
    n_vmodes : int
        Number of vmodes to simulate.
    z4_setpoint : float
        Z4 control setpoint in microns.
    zk_column_type : str
        'deviation' (OPD minus intrinsic, default) or 'opd' (full OPD).
    use_per_image_rotation : bool
        If True, recompute SVD per image at actual rotation_angle.
        When True, s_k varies per image; when False, s_k is fixed.

    Returns
    -------
    result_table : DataFrame
        Copy of input table with added columns (1-indexed labels):
        precon_vmode_k, precon_pv_k, precon_mv_k, precon_mv_p_k, precon_mv_i_k,
        precon_pv_used_k, precon_delta_dof_vmode_k,
        z_corrected_{corner}, z_intrinsic_{corner} (per corner).
    """
    if max_integral is None:
        max_integral = np.ones(n_vmodes) * 5.0
    max_integral = np.atleast_1d(max_integral)

    # Broadcast scalar gains to per-vmode arrays
    kappaP = np.broadcast_to(np.atleast_1d(kappaP), (n_vmodes,)).copy()
    kappaI = np.broadcast_to(np.atleast_1d(kappaI), (n_vmodes,)).copy()
    kappaD = np.broadcast_to(np.atleast_1d(kappaD), (n_vmodes,)).copy()

    seqs = table["seq"].values.astype(int)
    seq_num_corr = table["seq_num_corr"].values.astype(int)
    n_exp = len(table)

    # Compute vmode amplitudes from Zernike columns (measurement-space: a_k = U_k^T z)
    # Returns per-image singular values when use_per_image_rotation=True
    vmode_from_zk, z_corrected_all, s_per_image = table_to_vmode_timeseries(
        table, svd_result, zn_idx, n_vmodes,
        z4_setpoint=z4_setpoint, zk_column_type=zk_column_type,
        use_per_image_rotation=use_per_image_rotation)

    # Extract DOF-based vmodes from table (ground truth corrections)
    vmodes_from_table = np.vstack(table['vmodes'].values)

    # Compute DOF-based vmode deltas (actual corrections applied)
    delta_dof_vmode = np.zeros((n_exp, n_vmodes))
    delta_dof_vmode[1:] = vmodes_from_table[1:, :n_vmodes] - vmodes_from_table[:-1, :n_vmodes]

    # Build seq -> index lookup
    seq_to_idx = {int(s_val): i for i, s_val in enumerate(seqs)}

    # Initialize output arrays
    pv = np.zeros((n_exp, n_vmodes))
    mv = np.zeros((n_exp, n_vmodes))
    mv_p = np.zeros((n_exp, n_vmodes))
    mv_i = np.zeros((n_exp, n_vmodes))
    mv_d = np.zeros((n_exp, n_vmodes))
    pv_used = np.zeros((n_exp, n_vmodes), dtype=bool)

    # Run PID for each vmode independently
    for k in range(n_vmodes):
        integral = 0.0
        prev_pv = 0.0
        used_seq_nums = set()

        for i in range(n_exp):
            corr_seq = int(seq_num_corr[i])

            if corr_seq in seq_to_idx and corr_seq not in used_seq_nums:
                corr_idx = seq_to_idx[corr_seq]
                # Scale by 1/s_k to match Rubin's DOF-space vmode convention
                # Rubin: z → dof_state → get_vmodes_from_dofs → v_k = (U_k^T z) / s_k
                # When use_per_image_rotation=True, s_k varies per image
                pv_value = vmode_from_zk[corr_idx, k] / s_per_image[corr_idx, k]

                if not np.isnan(pv_value):
                    pv[i, k] = pv_value
                    pv_used[i, k] = True
                    used_seq_nums.add(corr_seq)

                    mv[i, k], mv_p[i, k], mv_i[i, k], mv_d[i, k], integral = \
                        _pid_step(pv_value, kappaP[k], kappaI[k], kappaD[k],
                                  integral, prev_pv, max_integral[k])
                    prev_pv = pv_value
                else:
                    mv_i[i, k] = kappaI[k] * integral
                    mv[i, k] = mv_i[i, k]
            else:
                mv_i[i, k] = kappaI[k] * integral
                mv[i, k] = mv_i[i, k]

    # Build output table (1-indexed vmode labels)
    result = table.copy()
    for k in range(n_vmodes):
        label = k + 1  # 1-indexed
        result[f"precon_vmode_{label}"] = vmode_from_zk[:, k]
        result[f"precon_pv_{label}"] = pv[:, k]
        result[f"precon_mv_{label}"] = mv[:, k]
        result[f"precon_mv_p_{label}"] = mv_p[:, k]
        result[f"precon_mv_i_{label}"] = mv_i[:, k]
        result[f"precon_pv_used_{label}"] = pv_used[:, k]
        result[f"precon_delta_dof_vmode_{label}"] = delta_dof_vmode[:, k]

    # Add per-corner corrected Zernike columns (after setpoint correction)
    for corner in wf_corners:
        result[f"z_corrected_{corner}"] = z_corrected_all[corner]

    # Copy per-corner intrinsic Zernikes from input table (Butler data)
    for corner in wf_corners:
        col = f"zk_intrinsic_{corner}"
        if col in table.columns:
            result[f"z_intrinsic_{corner}"] = table[col].values

    # Also store DOF-space corrections
    # MV is already in 1/s-scaled vmode space, so convert directly:
    # dof = norm * V @ mv  (no additional /s needed)
    V = svd_result["V"]
    norm_sub = svd_result["norm_vector"]  # already subset-restricted to active DOFs
    dof_corrections = np.zeros((n_exp, len(svd_result["dof_indices"])))
    for i in range(n_exp):
        mv_vec = mv[i, :min(n_vmodes, V.shape[1])]
        q = V[:, :len(mv_vec)] @ mv_vec
        dof_corrections[i] = norm_sub * q
    result["precon_dof_correction"] = list(dof_corrections)

    print(f"PID simulation complete: {n_exp} exposures, {n_vmodes} vmodes")
    print(f"  Z4_setpoint = {z4_setpoint} um")
    print(f"  Input: zk_{zk_column_type} columns (PV scaled by 1/sigma_k)")
    if use_per_image_rotation:
        print(f"  Per-image rotation SVD: s_k varies per exposure")
    print(f"  Per-vmode gains:")
    for k in range(n_vmodes):
        label = k + 1
        pv_range = np.nanmax(np.abs(pv[:, k]))
        mv_range = np.nanmax(np.abs(mv[:, k]))
        n_used = pv_used[:, k].sum()
        print(f"    Vmode {label}: kP={kappaP[k]:.3f} kI={kappaI[k]:.3f} kD={kappaD[k]:.3f} "
              f"max_I={max_integral[k]:.3f} | |PV|_max={pv_range:.4f} |MV|_max={mv_range:.4f} "
              f"PV_used={n_used}/{n_exp}")

    return result

<a id="Load-Data"></a>
## Load Data

In [ ]:
# Load the nightly table
table = pd.read_parquet(parquet_file)
print(f'Loaded {parquet_file}: {len(table)} rows')
print(f'Columns: {sorted(table.columns.tolist())}')

# Verify Butler Zernike columns exist for the selected type
butler_cols = [f"zk_{zk_column_type}_{c}" for c in wf_corners]
missing = [c for c in butler_cols if c not in table.columns]
if missing:
    raise ValueError(f"Missing Butler Zernike columns: {missing}. "
                     f"Regenerate parquet with updated nightly_tablemaker.")
print(f'Butler Zernike columns present ({zk_column_type}): {butler_cols}')

# Check for NaN coverage
for col in butler_cols:
    n_valid = table[col].dropna().shape[0]
    print(f'  {col}: {n_valid}/{len(table)} valid')

# Extract key arrays
seqs = table['seq'].values.astype(int)
seq_num_corr = table['seq_num_corr'].values.astype(int)
dof_state = np.vstack(table['dof_state'].values)
vmodes_from_table = np.vstack(table['vmodes'].values)

# Build sensitivity matrix and SVD
svd_result = build_sensitivity_matrix(dof_set_name, dof_sets, zn_idx, sensor_name_list, ofc_config_dir)

# --- Diagnose why signs differ from StateEstimator ---
print("\n=== SVD Matrix Dimension Comparison ===")
print(f"Our notebook: {len(zn_idx)} Zernikes x {len(sensor_name_list)} sensors = {len(zn_idx)*len(sensor_name_list)} rows, "
      f"{len(dof_sets[dof_set_name])} DOFs ({dof_set_name})")
try:
    ofc_default = OFCData()
    ofc_default.configure_controller()
    n_active_default = len(ofc_default.dof_idx)
    n_zk_default = ofc_default.znmax - ofc_default.znmin + 1
    n_sensors = len(sensor_name_list)
    print(f"StateEstimator (OFCData default): {n_zk_default} Zernikes x {n_sensors} sensors = {n_zk_default*n_sensors} rows, "
          f"{n_active_default} active DOFs (dof_idx = {list(ofc_default.dof_idx)})")
    print(f"  znmin={ofc_default.znmin}, znmax={ofc_default.znmax}")
    print(f"=> Different matrix dimensions explain the SVD sign ambiguity (8/12 flips is normal)")
except Exception as e:
    print(f"  (Could not create default OFCData to compare: {e})")

# Align SVD signs with StateEstimator convention
print("\nSign alignment:")
align_svd_signs(svd_result, dof_state, vmodes_from_table, n_vmodes)

# Verify DOF-based vmodes match after alignment
V = svd_result["V"]
norm_sub = svd_result["norm_vector"]  # already subset-restricted to active DOFs
dof_idx = svd_result["dof_indices"]
vmodes_from_dof_ours = np.zeros((len(table), n_vmodes))
for i in range(len(table)):
    dof_used = dof_state[i, dof_idx]
    vmodes_from_dof_ours[i] = (V.T @ (dof_used / norm_sub))[:n_vmodes]

print(f'\nPost-alignment verification (DOF-based vmodes, table vs ours):')
for k in range(n_vmodes):
    corr = np.corrcoef(vmodes_from_table[:, k], vmodes_from_dof_ours[:, k])[0, 1]
    ratio = np.std(vmodes_from_table[:, k]) / max(np.std(vmodes_from_dof_ours[:, k]), 1e-20)
    print(f'  Vmode {k+1}: correlation = {corr:.6f}, std_ratio = {ratio:.4f}')

print(f"\nZ4_setpoint = {Z4_setpoint} um")
print(f"Zernike source = zk_{zk_column_type}")

In [ ]:
# 4x3 panel: Our DOF-based vmodes vs Table vmodes (post sign-alignment)
fig, axes = plt.subplots(4, 3, figsize=(16, 18))
fig.suptitle("Post-alignment: Our SVD vs Table Vmodes (DOF-based)", fontsize=14, y=1.01)

for k in range(n_vmodes):
    ax = axes[k // 3, k % 3]
    label = k + 1

    x = vmodes_from_table[:, k]
    y = vmodes_from_dof_ours[:, k]
    corr = np.corrcoef(x, y)[0, 1]

    sc = ax.scatter(x, y, c=seqs, s=6, alpha=0.6, cmap='viridis')
    # y=x reference line
    lim_min = min(x.min(), y.min())
    lim_max = max(x.max(), y.max())
    margin = (lim_max - lim_min) * 0.05
    ax.plot([lim_min - margin, lim_max + margin],
            [lim_min - margin, lim_max + margin], 'r--', lw=1)
    ax.set_xlim(lim_min - margin, lim_max + margin)
    ax.set_ylim(lim_min - margin, lim_max + margin)
    ax.set_aspect('equal')
    ax.set_title(f"Vmode {label}  (r = {corr:.6f})", fontsize=10)
    ax.set_xlabel('Table (StateEstimator)', fontsize=8)
    ax.set_ylabel('Our SVD', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print all 12 correlations in a compact summary
print("Vmode correlation summary (table vs our SVD, post-alignment):")
for k in range(n_vmodes):
    corr = np.corrcoef(vmodes_from_table[:, k], vmodes_from_dof_ours[:, k])[0, 1]
    print(f"  Vmode {k+1}: r = {corr:.6f}")

In [ ]:
# =====================================================================
# Vmode Orthogonality Analysis: Our SVD vs StateEstimator
# =====================================================================
# Our SVD (84x22): V columns are exactly orthonormal in standard_22 space.
# StateEstimator now also uses standard_22 (via comp_dof_idx), so both SVDs
# should produce the same basis. This cell validates that.

# --- 1. Replicate StateEstimator's SVD (matching nightly_tablemaker with comp_dof_idx) ---
ofc_se = OFCData()
ofc_se.configure_controller()
await ofc_se.configure_instrument("lsst")

# Restrict to standard_22 DOFs (matching nightly_tablemaker)
m2_hexapod = np.ones(5, dtype=bool)
cam_hexapod = np.ones(5, dtype=bool)
m1m3_bending = np.zeros(20, dtype=bool)
m2_bending = np.zeros(20, dtype=bool)
m1m3_bending[:7] = True
m2_bending[:5] = True
ofc_se.comp_dof_idx = dict(
    m2HexPos=m2_hexapod,
    camHexPos=cam_hexapod,
    M1M3Bend=m1m3_bending,
    M2Bend=m2_bending,
)

se = StateEstimator(ofc_se)

field_angles_se = [ofc_se.sample_points[s] for s in sensor_name_list]
sens_mat_se = se.get_sensitivity_matrix(
    field_angles_se, 0.0, normalize=True, truncate=False, check_invertible=False)

U_se, s_se, Vh_se = np.linalg.svd(sens_mat_se, full_matrices=False)
V_se = Vh_se.T  # shape: (n_active_dofs_se, n_active_dofs_se)

active_dofs_se = list(ofc_se.dof_idx)
n_active_se = len(active_dofs_se)

print(f"=== Matrix Dimensions ===")
print(f"Our SVD:          {svd_result['Atilde_sub'].shape}  ({len(zn_idx)} Zernikes x {len(sensor_name_list)} sensors, {len(dof_sets[dof_set_name])} DOFs)")
print(f"StateEstimator:   {sens_mat_se.shape}  (SE Zernikes x {len(sensor_name_list)} sensors, {n_active_se} active DOFs)")
print(f"Our V shape:      {svd_result['V'].shape}")
print(f"StateEstimator V: {V_se.shape}")
print(f"SE active DOFs:   {active_dofs_se}")

# --- 2. Map standard_22 indices into StateEstimator's active DOF space ---
std22_dofs = dof_sets["standard_22"]
std22_in_active = [active_dofs_se.index(d) for d in std22_dofs if d in active_dofs_se]
n_mapped = len(std22_in_active)
print(f"\nStandard_22 DOFs mapped into SE active space: {n_mapped}/{len(std22_dofs)}")

# --- 3. Compute Gram matrices (first 12 modes) ---
n_compare = min(n_vmodes, V_se.shape[1])
V_ours = svd_result["V"][:, :n_compare]
G_ours = V_ours.T @ V_ours  # Should be I exactly

V_se_proj = V_se[std22_in_active, :n_compare]  # Project SE modes onto standard_22
G_se_proj = V_se_proj.T @ V_se_proj  # Should also be ~I now with comp_dof_idx

# --- 4. Per-mode fraction in standard_22 ---
frac_in_std22 = np.array([np.sum(V_se[std22_in_active, k]**2) for k in range(n_compare)])

# --- 5. Plot ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Vmode Orthogonality: Our SVD vs StateEstimator (with comp_dof_idx)", fontsize=14)

# Panel 1: Our Gram matrix
ax = axes[0, 0]
im = ax.imshow(G_ours, cmap='RdBu_r', vmin=-0.1, vmax=1.1, aspect='equal')
ax.set_title("Our SVD: Gram matrix (should be I)")
ax.set_xlabel("Vmode")
ax.set_ylabel("Vmode")
ax.set_xticks(range(n_compare))
ax.set_xticklabels(range(1, n_compare+1), fontsize=8)
ax.set_yticks(range(n_compare))
ax.set_yticklabels(range(1, n_compare+1), fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)

# Panel 2: StateEstimator projected Gram matrix
ax = axes[0, 1]
im = ax.imshow(G_se_proj, cmap='RdBu_r', vmin=-0.1, vmax=1.1, aspect='equal')
ax.set_title("StateEstimator: Gram matrix (projected to std_22)")
ax.set_xlabel("Vmode")
ax.set_ylabel("Vmode")
ax.set_xticks(range(n_compare))
ax.set_xticklabels(range(1, n_compare+1), fontsize=8)
ax.set_yticks(range(n_compare))
ax.set_yticklabels(range(1, n_compare+1), fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)

# Panel 3: Non-orthogonality map |G_se - I|
ax = axes[1, 0]
non_orth = np.abs(G_se_proj - np.eye(n_compare))
im = ax.imshow(non_orth, cmap='hot_r', vmin=0, aspect='equal')
ax.set_title("|G_SE - I|: Non-orthogonality map")
ax.set_xlabel("Vmode")
ax.set_ylabel("Vmode")
ax.set_xticks(range(n_compare))
ax.set_xticklabels(range(1, n_compare+1), fontsize=8)
ax.set_yticks(range(n_compare))
ax.set_yticklabels(range(1, n_compare+1), fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)

# Panel 4: Fraction of mode content in standard_22
ax = axes[1, 1]
colors = ['C0' if f > 0.95 else 'C1' if f > 0.8 else 'C3' for f in frac_in_std22]
ax.bar(range(1, n_compare+1), frac_in_std22, color=colors, edgecolor='black', alpha=0.7)
ax.set_xlabel("Vmode")
ax.set_ylabel("Fraction in standard_22")
ax.set_title("SE vmode content in standard_22 subspace")
ax.set_xticks(range(1, n_compare+1))
ax.set_ylim(0, 1.05)
ax.axhline(1.0, color='red', ls='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='y')
for k, f in enumerate(frac_in_std22):
    ax.text(k+1, f, f'{f:.3f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

# --- 6. Print summary ---
off_diag_ours = np.max(np.abs(G_ours - np.eye(n_compare)))
off_diag_se = np.max(np.abs(G_se_proj - np.eye(n_compare)))
# Off-diagonal only (exclude diagonal):
mask_offdiag = ~np.eye(n_compare, dtype=bool)
max_offdiag_se = np.max(np.abs(G_se_proj[mask_offdiag]))

print(f"\n=== Orthogonality Summary ===")
print(f"Our SVD:          max |G - I| = {off_diag_ours:.2e}  (should be machine epsilon)")
print(f"StateEstimator:   max |G - I| = {off_diag_se:.4f}")
print(f"                  max off-diagonal = {max_offdiag_se:.4f}")

print(f"\nPer-mode fraction of SE vmode content in standard_22:")
for k in range(n_compare):
    leak = 1 - frac_in_std22[k]
    print(f"  Vmode {k+1}: {frac_in_std22[k]:.4f}  (leak to non-std22: {leak:.4f})")

print(f"\nSingular value comparison:")
print(f"  {'Vmode':<8} {'Ours':>10} {'SE':>10} {'Ratio':>10}")
for k in range(n_compare):
    ratio = svd_result['s'][k] / s_se[k] if s_se[k] > 0 else np.nan
    print(f"  {k+1:<8} {svd_result['s'][k]:>10.4f} {s_se[k]:>10.4f} {ratio:>10.4f}")

In [ ]:
# =====================================================================
# Z4-average vs delta_vmode_5 diagnostic
# Vmode 5 is dominated by Z4 (focus), so the correction should track Z4
# =====================================================================

# Compute Z4 averaged over 4 WFS corners (from full OPD, not deviation)
z4_avg = np.zeros(len(table))
for i in range(len(table)):
    z4_vals = []
    for corner in wf_corners:
        col = f"zk_opd_{corner}"
        zk = table.iloc[i][col]
        if zk is not None and not (isinstance(zk, float) and np.isnan(zk)):
            z4_vals.append(np.array(zk, dtype=float)[0])  # Z4 is index 0
    if z4_vals:
        z4_avg[i] = np.mean(z4_vals)

# Delta of DOF-based vmode 5 (1-indexed, so array index 4)
delta_dof_vmode5 = np.zeros(len(table))
delta_dof_vmode5[1:] = vmodes_from_table[1:, 4] - vmodes_from_table[:-1, 4]

# Z4 at the seq_num_corr exposure (what the controller saw when making the correction)
seq_to_idx_diag = {int(s): i for i, s in enumerate(seqs)}
z4_at_corr = np.full(len(table), np.nan)
for i in range(len(table)):
    corr_seq = int(seq_num_corr[i])
    if corr_seq in seq_to_idx_diag:
        z4_at_corr[i] = z4_avg[seq_to_idx_diag[corr_seq]]

# Quality mask for scatter: exclude first exposure and sequence gaps
diag_mask = np.ones(len(table), dtype=bool)
diag_mask[0] = False
diag_mask &= np.isfinite(z4_at_corr)
diag_mask[1:] &= np.diff(seqs) <= 2

# Plot
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Vmode 5 Validation: Z4-avg drives the focus correction", fontsize=14)

ax = axes[0, 0]
ax.scatter(seqs, z4_avg, s=8, alpha=0.6, label='Z4 avg (4 WFS corners, OPD)')
ax.set_ylabel('Z4 average (um wavefront)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title('Z4 average — measured wavefront focus')

ax = axes[0, 1]
ax.scatter(seqs, delta_dof_vmode5, s=8, alpha=0.6, color='C1',
           label='delta(vmode 5) from DOF state')
ax.set_ylabel('Delta vmode 5 (DOF-based)')
ax.set_xlabel('Sequence number')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title('Actual correction applied (DOF-based vmode 5 delta)')

ax = axes[1, 0]
ax.scatter(z4_at_corr[diag_mask], delta_dof_vmode5[diag_mask], s=8, alpha=0.5)
valid = diag_mask & np.isfinite(z4_at_corr)
if valid.sum() > 2:
    corr_val = np.corrcoef(z4_at_corr[valid], delta_dof_vmode5[valid])[0, 1]
else:
    corr_val = np.nan
ax.set_xlabel('Z4_avg at seq_num_corr (um)')
ax.set_ylabel('delta(vmode 5) (DOF-based)')
ax.set_title(f'Z4(corr) vs delta_vmode5  (correlation = {corr_val:.4f})')
ax.grid(True, alpha=0.3)

# Also show vmode 5 amplitude from corrected Zernikes (what the controller sees)
vmode_from_zk_diag, _, _ = table_to_vmode_timeseries(
    table, svd_result, zn_idx, n_vmodes,
    z4_setpoint=Z4_setpoint, zk_column_type=zk_column_type)
ax = axes[1, 1]
ax.scatter(seqs, vmode_from_zk_diag[:, 4], s=8, alpha=0.6, color='C2',
           label=f'Vmode 5 from {zk_column_type} Zernikes')
ax.scatter(seqs, z4_avg * 0.5, s=8, alpha=0.4, color='C3',
           label='Z4_avg x 0.5 (approximate scale)')
ax.set_xlabel('Sequence number')
ax.set_ylabel('Vmode amplitude')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_title(f'Vmode 5 from {zk_column_type} Zernikes tracks Z4 average')

plt.tight_layout()
plt.show()

print(f"Correlation Z4_avg(corr) vs delta_vmode5: {corr_val:.4f}")
print(f"(Expected: negative, because positive Z4 -> negative correction)")

<a id="PID-Simulation"></a>
## PID Simulation

Run independent PID loops on each vmode to reconstruct closed-loop behavior.

In [ ]:
result = simulate_pid_all_vmodes(
    table, svd_result, zn_idx,
    kappaP=kappaP, kappaI=kappaI, kappaD=kappaD,
    max_integral=max_integral, n_vmodes=n_vmodes,
    z4_setpoint=Z4_setpoint, zk_column_type=zk_column_type,
    use_per_image_rotation=use_per_image_rotation,
)

<a id="Validation-Plots"></a>
## Validation Plots

Compare simulated PID output with actual vmode changes for each vmode.

In [ ]:
# Detect sequence gaps and filter changes for quality masking
seq_gap = np.zeros(len(seqs), dtype=bool)
seq_gap[1:] = np.diff(seqs) > 1
seq_gap[0] = True

corr_gap = np.zeros(len(seqs), dtype=bool)
seq_set = set(seqs.astype(int))
corr_gap = np.array([int(s) not in seq_set for s in seq_num_corr])

bands = result['band'].values if 'band' in result.columns else None
filter_change = np.zeros(len(seqs), dtype=bool)
if bands is not None:
    filter_change[1:] = bands[1:] != bands[:-1]

blocks = result['block'].values if 'block' in result.columns else None

good_mask = ~seq_gap & ~corr_gap & ~filter_change
print(f'Quality mask: {good_mask.sum()}/{len(good_mask)} good exposures')

s = svd_result["s"]

# Collect per-vmode data for the combined figure
combined_predicted = []
combined_actual = []
combined_corr = []

for k in range(n_vmodes):
    label = k + 1  # 1-indexed
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f"Vmode {label} Validation (kP={kappaP[k]:.3f}, kI={kappaI[k]:.3f}, "
                 f"sigma_{label}={s[k]:.3f})", fontsize=14)

    pv_k = result[f"precon_pv_{label}"].values
    mv_k = result[f"precon_mv_{label}"].values
    mv_p_k = result[f"precon_mv_p_{label}"].values
    mv_i_k = result[f"precon_mv_i_{label}"].values
    delta_dof_k = result[f"precon_delta_dof_vmode_{label}"].values
    vmode_k = result[f"precon_vmode_{label}"].values

    # Predicted DOF-based vmode change: MV is already in 1/s-scaled space
    predicted_delta = mv_k

    # Panel 1: PV and MV vs sequence
    ax = axes[0, 0]
    ax.scatter(seqs, pv_k, s=5, alpha=0.5, label='PV (from Zernikes)')
    ax.scatter(seqs, mv_k, s=5, alpha=0.5, label='MV')
    ax.set_ylabel('PV / MV')
    ax.set_xlabel('Sequence')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_title(f"Vmode {label}: PV and MV vs sequence")

    # Panel 2: Actual vs predicted DOF-based vmode delta
    ax = axes[0, 1]
    ax.scatter(seqs, delta_dof_k, s=5, alpha=0.5, label='Actual delta(dof vmode)')
    ax.scatter(seqs, predicted_delta, s=5, alpha=0.5, label='Predicted: MV')
    ylim = np.nanpercentile(np.abs(delta_dof_k[good_mask]), 99) * 2
    if ylim > 0:
        ax.set_ylim(-ylim, ylim)
    ax.set_ylabel('Delta DOF vmode')
    ax.set_xlabel('Sequence')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_title(f"Vmode {label}: Actual vs Predicted correction")

    # Panel 3: Correlation scatter (robust biweight midcorrelation)
    ax = axes[1, 0]
    gm = good_mask
    if blocks is not None:
        for bname in np.unique(blocks[gm]):
            bmask = gm & (blocks == bname)
            ax.scatter(predicted_delta[bmask], delta_dof_k[bmask], s=5, alpha=0.5, label=bname)
    else:
        ax.scatter(predicted_delta[gm], delta_dof_k[gm], s=5, alpha=0.5)
    lim = max(np.nanpercentile(np.abs(predicted_delta[gm]), 99),
             np.nanpercentile(np.abs(delta_dof_k[gm]), 99)) * 1.5
    if lim > 0:
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
    ax.axline((0, 0), slope=1, color='red', linestyle='--', linewidth=1.5, label='y=x')
    ax.set_xlabel('Predicted: MV')
    ax.set_ylabel('Actual: delta(dof vmode)')
    ax.set_aspect('equal')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    corr_val = biweight_midcorrelation(predicted_delta[gm], delta_dof_k[gm])
    ax.set_title(f"Vmode {label}: Biweight corr (r={corr_val:.4f})")

    # Panel 4: Residual histogram
    ax = axes[1, 1]
    residual = delta_dof_k[gm] - predicted_delta[gm]
    if len(residual) > 0 and np.any(np.isfinite(residual)):
        mad = median_abs_deviation(residual[np.isfinite(residual)], scale='normal')
        ax.hist(residual[np.isfinite(residual)], bins=80, edgecolor="black", alpha=0.7)
        ax.set_title(f"Vmode {label}: Residual (MAD={mad:.5f})")
    else:
        ax.set_title(f"Vmode {label}: Residual (no data)")
    ax.set_xlabel('Residual (actual - predicted)')
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Store data for combined figure
    combined_predicted.append(predicted_delta[gm])
    combined_actual.append(delta_dof_k[gm])
    combined_corr.append(corr_val)

# --- Save combined validation figure (3x4 grid of correlation scatters) ---
fig_combined, axes_combined = plt.subplots(3, 4, figsize=(20, 15))
fig_combined.suptitle(f"Vmode Validation — {day_obs} (zk_{zk_column_type})", fontsize=16)

for k in range(n_vmodes):
    ax = axes_combined.flat[k]
    label = k + 1
    pred = combined_predicted[k]
    actual = combined_actual[k]
    corr = combined_corr[k]

    ax.scatter(pred, actual, s=5, alpha=0.5)
    lim = max(np.nanpercentile(np.abs(pred), 99),
              np.nanpercentile(np.abs(actual), 99)) * 1.5
    if lim > 0:
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
    ax.axline((0, 0), slope=1, color='red', linestyle='--', linewidth=1.5)
    ax.set_aspect('equal')
    ax.set_title(f"Vmode {label} (r={corr:.3f})", fontsize=11)
    ax.set_xlabel('Predicted (MV)', fontsize=8)
    ax.set_ylabel('Actual Δdof_vmode', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

fig_combined.tight_layout()
validation_file = f"vmode_validation_{day_obs}.png"
fig_combined.savefig(validation_file, dpi=150, bbox_inches='tight')
plt.close(fig_combined)
print(f"Saved: {validation_file}")

In [ ]:
# Summary: correlation and MAD of residuals per vmode
mads = []
corrs = []
for k in range(n_vmodes):
    label = k + 1
    delta_k = result[f"precon_delta_dof_vmode_{label}"].values
    mv_k = result[f"precon_mv_{label}"].values
    # MV is already in 1/s-scaled space (no additional /s needed)
    predicted = mv_k
    residual = delta_k[good_mask] - predicted[good_mask]
    finite = residual[np.isfinite(residual)]
    if len(finite) > 0:
        mads.append(median_abs_deviation(finite, scale='normal'))
        corrs.append(biweight_midcorrelation(predicted[good_mask], delta_k[good_mask]))
    else:
        mads.append(np.nan)
        corrs.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.bar(range(1, n_vmodes+1), corrs, edgecolor="black", alpha=0.7)
ax.set_xlabel('Vmode')
ax.set_ylabel('Biweight Midcorrelation')
ax.set_title('PID Prediction vs Actual Correction: Robust Correlation per Vmode')
ax.set_xticks(range(1, n_vmodes+1))
ax.set_ylim(-0.1, 1.1)
ax.axhline(1.0, color='red', ls='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis="y")
for i_k, c in enumerate(corrs):
    if np.isfinite(c):
        ax.text(i_k+1, c, f'{c:.3f}', ha='center', va='bottom', fontsize=8)

ax = axes[1]
ax.bar(range(1, n_vmodes+1), mads, edgecolor="black", alpha=0.7)
ax.set_xlabel('Vmode')
ax.set_ylabel('MAD (sigma-equivalent) of residual')
ax.set_title('PID Simulation Accuracy per Vmode')
ax.set_xticks(range(1, n_vmodes+1))
ax.grid(True, alpha=0.3, axis="y")
for i_k, m in enumerate(mads):
    if np.isfinite(m):
        ax.text(i_k+1, m, f'{m:.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print('\nSummary (biweight midcorrelation):')
for k in range(n_vmodes):
    print(f'  Vmode {k+1}: correlation = {corrs[k]:.4f}, MAD = {mads[k]:.6f}')

In [ ]:
# Save augmented table
outfile = f"aos_openloop_{day_obs}.parquet"
result.to_parquet(outfile)
print(f"Saved {outfile}")


<a id="Multi-Night"></a>
## Multi-Night

Uncomment to run over multiple nights.

In [ ]:
# day_obs_list = [20260115, 20260116, 20260117]
# for d in day_obs_list:
#     t = pd.read_parquet(f"nightly_aos_table_{d}.parquet")
#     r = simulate_pid_all_vmodes(t, svd_result, zn_idx,
#                                 kappaP=kappaP, kappaI=kappaI, kappaD=kappaD,
#                                 max_integral=max_integral, n_vmodes=n_vmodes,
#                                 z4_setpoint=Z4_setpoint,
#                                 zk_column_type=zk_column_type)
#     r.to_parquet(f"aos_openloop_{d}.parquet")
#     print(f"Done {d}: {len(r)} rows")